In [1]:
# Construccion de la base

import sys
sys.path.append("..")

import re
import sqlite3
import pandas as pd

from src.data import cargar_enemdu
from src.features import agregar_variables

df = agregar_variables(cargar_enemdu())

cols = ["fexp", "condact", "p02", "sexo", "zona", "dominio", "dominio_nombre", "p03",
        "grupo_edad", "nnivins", "anios_estudio", "secemp", "rama1", "grupo1",
        "ingrl", "horas_total", "etnia", "upm"]
tabla = df[cols].copy()
tabla["grupo_edad"] = tabla["grupo_edad"].astype("object")    # SQLite no maneja categorías de pandas

dim_condact = pd.DataFrame({
    "codigo": range(10),
    "nombre": ["Menores de 15", "Empleo adecuado", "Subempleo por tiempo", "Subempleo por ingresos",
               "Otro empleo no pleno", "Empleo no remunerado", "Empleo no clasificado",
               "Desempleo abierto", "Desempleo oculto", "Inactivos"],
})

ramas = {1: "Agricultura y pesca", 2: "Minas y canteras", 3: "Manufactura", 4: "Electricidad y gas",
         5: "Agua y alcantarillado", 6: "Construcción", 7: "Comercio", 8: "Transporte",
         9: "Alojamiento y comidas", 10: "Información y comunic.", 11: "Finanzas y seguros",
         12: "Inmobiliarias", 13: "Prof. y científicas", 14: "Serv. administrativos",
         15: "Administración pública", 16: "Enseñanza", 17: "Salud y serv. sociales",
         18: "Artes y recreación", 19: "Otros servicios", 20: "Hogares (domést.)",
         21: "Organizaciones extraterr.", 22: "No especificado"}
dim_rama = pd.DataFrame({"codigo": list(ramas.keys()), "nombre": list(ramas.values())})

con = sqlite3.connect("../data/processed/enemdu.db")
tabla.to_sql("personas", con, if_exists="replace", index=False)
dim_condact.to_sql("dim_condact", con, if_exists="replace", index=False)
dim_rama.to_sql("dim_rama", con, if_exists="replace", index=False)

print(pd.read_sql_query("SELECT COUNT(*) AS filas, ROUND(SUM(fexp)) AS poblacion FROM personas", con))

   filas   poblacion
0  82894  19026779.0


In [2]:
# Ejecucion de las consultas

texto = open("../sql/queries.sql", encoding="utf-8").read()
consultas = {m.group(1): m.group(2).strip().rstrip(";")
             for m in re.finditer(r"-- name: (\w+)\n(.*?)(?=\n-- name:|\Z)", texto, re.S)}
print(list(consultas))

for nombre, sql in consultas.items():
    print(f"\n=== {nombre} ===")
    print(pd.read_sql_query(sql, con).to_string(index=False))

['q1_nacional', 'q2_dominios', 'q3_informalidad_zona_sexo', 'q4_pareto_rama', 'q5_cambio_por_edad', 'q6_mediana_ponderada']

=== q1_nacional ===
 participacion  empleo_adecuado  subempleo  desempleo
          63.7             35.7       18.4        3.4

=== q2_dominios ===
dominio_nombre  participacion  empleo_adecuado  subempleo  desempleo
        Ambato           69.0             43.9       17.0        3.7
        Cuenca           60.8             60.7        7.7        4.0
     Guayaquil           58.3             46.1       22.6        2.3
       Machala           62.7             49.1       14.3        3.8
         Quito           61.3             55.7       11.5        8.9

=== q3_informalidad_zona_sexo ===
  zona   sexo  pct_informal
Urbana Hombre          44.7
Urbana  Mujer          37.2
 Rural Hombre          73.5
 Rural  Mujer          78.9

=== q4_pareto_rama ===
                     rama  pct_informal  pct_de_los_informales  acumulado
      Agricultura y pesca          86.1